In [5]:
import pandas as pd
import wbgapi as wb
import atoti as tt
import numpy as np
from functools import reduce

Welcome to Atoti 0.9.15!

By using this community edition, you agree with the license available at https://docs.activeviam.com/products/atoti/python-sdk/latest/eula.html.
Browse the official documentation at https://docs.activeviam.com/products/atoti/python-sdk.
Join the community at https://www.atoti.io/register.

Atoti collects telemetry data, which is used to help understand how to improve the product.
If you don't wish to send usage data, you can request a trial license at https://www.atoti.io/evaluation-license-request.

You can hide this message by setting the `ATOTI_HIDE_EULA_MESSAGE` environment variable to True.


In [7]:
def get_wb_data(name, code):
    df = wb.data.DataFrame([code], ['IND'])
    df = df.transpose().reset_index()

    df['index'] = df['index'].str.replace("YR", "")
    df['index'] = pd.to_datetime(df['index'])
    
    df.rename(
        columns={
            'index': 'Year',
            'IND': name
        },
        inplace=True
    )

    df[name] = df[name].astype(float)

    return df

In [ ]:
NY.GDP.MKTP.CD

In [8]:
data_codes = {
    "GDP": "NY.GDP.MKTP.CD",
    'Gross Capital Formation':'NE.GDI.TOTL.CD',
    "Final consumption expenditure":'NE.CON.TOTL.CD',
    "Government final consumption expenditure":"NE.CON.GOVT.CD",
    "GDP per capita": "NY.GDP.PCAP.CD",
    "Manufacturing share of GDP":'NV.IND.MANF.ZS',
    "Industry share of GDP":"NV.IND.TOTL.ZS",
    "Services share of GDP":"NV.SRV.TOTL.ZS",
    "Agriculture share of GDP":"NV.AGR.TOTL.ZS",
    "FDI (net inflows in $)":'BX.KLT.DINV.CD.WD', 
    "Inflation consumer prices (annual %)":"FP.CPI.TOTL.ZG",
    "Unemployment rate":"SL.UEM.TOTL.ZS",
    "Labor force participation rate (ages 15-64)":"SL.TLF.ACTI.ZS",
}


# 1. Build all datasets
datasets = {
    name: get_wb_data(name, code)
    for name, code in data_codes.items()
}

# 2. Ensure GDP is the left table
gdp_df = datasets["GDP"].copy()

# All other indicators
other_dfs = [df for name, df in datasets.items() if name != "GDP"]

# 3. Left-join everything onto GDP (on "Year")
df = reduce(
    lambda left, right: pd.merge(left, right, on="Year", how="left"),
    other_dfs,
    gdp_df  # initial value = GDP dataframe
)

In [10]:
df['FDI (net inflows in $)']=df['FDI (net inflows in $)']/1000000000
df.rename(columns={'FDI (net inflows in $)':'FDI (net inflows in USD Bn)'}, inplace=True) 

In [11]:
df

economy,Year,GDP,Gross Capital Formation,Final consumption expenditure,Government final consumption expenditure,GDP per capita,Manufacturing share of GDP,Industry share of GDP,Services share of GDP,Agriculture share of GDP,FDI (net inflows in $),Inflation consumer prices (annual %),Unemployment rate,Labor force participation rate (ages 15-64)
0,1960-01-01,3.702988e+10,6.639953e+09,3.479232e+10,2.434156e+09,84.932808,14.750118,20.834343,38.782462,41.741335,NaN,1.779878,NaN,NaN
1,1961-01-01,3.923244e+10,7.557009e+09,3.654097e+10,2.704945e+09,87.853861,15.353836,21.434844,38.325875,41.092482,NaN,1.695213,NaN,NaN
2,1962-01-01,4.216148e+10,7.634420e+09,3.891523e+10,3.279987e+09,92.199958,15.863298,22.052600,39.935845,39.065969,NaN,3.632215,NaN,NaN
3,1963-01-01,4.842192e+10,9.197966e+09,4.377812e+10,4.215148e+09,103.435021,15.752388,21.879476,38.095180,39.825352,NaN,2.946161,NaN,NaN
4,1964-01-01,5.648029e+10,1.103050e+10,5.111781e+10,4.542440e+09,117.856431,14.850740,20.955289,36.340821,41.343717,NaN,13.355261,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
61,2021-01-01,3.167271e+12,1.017195e+12,2.262279e+12,3.318153e+11,2239.613844,14.377029,26.466524,47.847223,17.372562,4.472728e+10,5.131407,6.380,55.481
62,2022-01-01,3.249938e+12,1.118362e+12,2.216647e+12,3.597293e+11,2279.981457,13.332720,25.604026,47.172752,18.210666,4.994026e+10,6.699034,4.822,56.666
63,2023-01-01,3.500906e+12,1.207977e+12,2.349411e+12,3.712885e+11,2434.448263,13.312844,25.552556,47.586465,17.656539,2.808619e+10,5.649143,4.172,57.925
64,2024-01-01,3.760813e+12,1.291685e+12,2.526254e+12,4.014557e+11,2591.991661,13.139220,25.172263,47.970627,17.573580,2.713985e+10,4.953036,4.173,58.991


In [10]:
df.to_csv('total_data.csv')

In [12]:
import sdmx

IMF_DATA = sdmx.Client('IMF_DATA')

data_msg = IMF_DATA.data(
    'WEO',
    key='IND.NGSD_NGDP.A',
    params={'startPeriod': 1980}
)

df_savings = sdmx.to_pandas(data_msg)
df_savings=pd.DataFrame(df_savings)

xml.Reader got no structure=… argument for StructureSpecificData


In [13]:
df_savings=df_savings.reset_index()
df_savings=df_savings[['TIME_PERIOD','value']]
df_savings['TIME_PERIOD']=pd.to_datetime(df_savings['TIME_PERIOD'])
df_savings=df_savings[df_savings['TIME_PERIOD']<'2026-01-01']

In [14]:
sectorwise_emp=pd.read_csv(r'/Users/nakuliyer/Documents/Advaya work- local files/Own work- NI/Dashboard 1/local csv/Employment share data.csv')
sectorwise_emp.dropna(inplace=True)
sectorwise_emp=sectorwise_emp.replace("%","", regex=True)
sectorwise_emp[list(sectorwise_emp.columns[1:])]=sectorwise_emp[list(sectorwise_emp.columns[1:])].astype(float)

sectorwise_emp["Start_Date"] = pd.to_datetime(
    sectorwise_emp["Period"].str.split("-").str[0] + "-04-01"
)

In [18]:
non_food_credit=pd.read_csv(r'/Users/nakuliyer/Documents/Advaya work- local files/Own work- NI/Dashboard 1/local csv/Food & Non-Food Credit of Scheduled Commercial Banks.csv')

non_food_credit['date']=pd.to_datetime(non_food_credit['Fortnight Date Final'])


/var/folders/bn/y8_wsc6d5yd6vsxff4856qsr0000gn/T/ipykernel_10946/3218733162.py:3: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  non_food_credit['date']=pd.to_datetime(non_food_credit['Fortnight Date Final'])


In [19]:

# Parse the fortnight date, e.g. "31-Mar-26"
non_food_credit["date"] = pd.to_datetime(
    non_food_credit["Fortnight Date Final"],
    format="%d-%b-%y",
    errors="coerce"
)

# Clean the numeric columns
for col in ["Bank Credit", "Food Credit"]:
    non_food_credit[col] = pd.to_numeric(
        non_food_credit[col].astype(str).str.replace(",", "", regex=False),
        errors="coerce"
    )

# Recalculate NFC to avoid malformed rows in DBIE export
non_food_credit["non_food_credit"] = non_food_credit["Bank Credit"] - non_food_credit["Food Credit"]

# Retain valid outstanding-credit observations
non_food_credit = non_food_credit[
    non_food_credit["date"].notna()
    & non_food_credit["Bank Credit"].notna()
    & non_food_credit["Food Credit"].notna()
    & non_food_credit["non_food_credit"].gt(0)
].copy()

# A date in Jan-Mar belongs to FY ending in that calendar year.
# Example: 31-Mar-2026 -> FY 2025-26 / FY-end year 2026.
non_food_credit["fy_end_year"] = non_food_credit["date"].dt.year + (non_food_credit["date"].dt.month >= 4).astype(int)

# Keep only dates up to the relevant FY-end 31 March.
# This is inherently satisfied by assigning Apr-Dec to the following FY.
# Then select the latest valid observation per FY.
fy_end_nfc = (
    non_food_credit.sort_values("date")
      .groupby("fy_end_year", as_index=False)
      .tail(1)
      .sort_values("fy_end_year")
      [["fy_end_year", "date", "non_food_credit"]]
)

fy_end_nfc["financial_year"] = (
    "FY "
    + (fy_end_nfc["fy_end_year"] - 1).astype(str)
    + "-"
    + fy_end_nfc["fy_end_year"].astype(str).str[-2:]
)

fy_end_nfc = fy_end_nfc[
    ["financial_year", "date", "non_food_credit"]
].rename(columns={
    "date": "as_of_date",
    "non_food_credit": "non_food_credit_crore"
})

print(fy_end_nfc.to_string(index=False))

financial_year as_of_date  non_food_credit_crore
    FY 1997-98 1998-03-27               311594.0
    FY 1998-99 1999-03-26               352054.0
    FY 1999-00 2000-03-24               410267.0
    FY 2000-01 2001-03-23               471443.0
    FY 2001-02 2002-03-22               535745.0
    FY 2002-03 2003-03-21               679736.0
    FY 2003-04 2004-03-19               804824.0
    FY 2004-05 2005-03-18              1059308.0
    FY 2005-06 2006-03-31              1466387.0
    FY 2006-07 2007-03-30              1884669.0
    FY 2007-08 2008-03-28              2317515.0
    FY 2008-09 2009-03-27              2729338.0
    FY 2009-10 2010-03-26              3196299.0
    FY 2010-11 2011-03-25              3877801.0
    FY 2011-12 2012-03-23              4530549.0
    FY 2012-13 2013-03-22              5164038.0
    FY 2013-14 2014-03-21              5895649.0
    FY 2014-15 2015-03-20              6442002.0
    FY 2015-16 2016-03-18              7144362.0
    FY 2016-17 2017-

In [ ]:
fy_end_nfc['financial_year']=fy_end_nfc['financial_year'].str.replace("FY","")

In [22]:
fy_end_nfc['non_food_credit_crore']=fy_end_nfc['non_food_credit_crore']/100000

In [25]:
fy_end_nfc.rename(columns={'non_food_credit_crore':'non_food_credit_lakh_crore'}, inplace=True)

In [27]:
msme_emp_df=pd.read_csv(r'/Users/nakuliyer/Documents/Advaya work- local files/Own work- NI/Dashboard 1/local csv/MSME employment data.csv')
msme_emp_df['Total employees working in MSMEs registered on Udyam portal']=msme_emp_df['Total employees working in MSMEs registered on Udyam portal'].str.replace(",","")



In [28]:
msme_emp_df

,Period,Total employees working in MSMEs registered on Udyam portal
0,2021-22,34953245
1,2022-23,46035185
2,2023-24,74455733


In [29]:
repo_rates=pd.read_csv(r'/Users/nakuliyer/Documents/Advaya work- local files/Own work- NI/Dashboard 1/local csv/Repo rates.csv')

In [31]:
repo_rates['Dates']=pd.to_datetime(repo_rates['Dates'])

/var/folders/bn/y8_wsc6d5yd6vsxff4856qsr0000gn/T/ipykernel_10946/814271667.py:1: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  repo_rates['Dates']=pd.to_datetime(repo_rates['Dates'])


In [32]:
repo_rates=repo_rates.dropna()

In [34]:
NPAs=pd.read_csv(r'/Users/nakuliyer/Documents/Advaya work- local files/Own work- NI/Dashboard 1/local csv/NPAs.csv')

In [35]:
market_cap=pd.read_csv(r'/Users/nakuliyer/Documents/Advaya work- local files/Own work- NI/Dashboard 1/local csv/Market Capitalisations.csv')

In [38]:
market_cap

,End-period,Market Capitalisation - NSE,Market Capitalisation - BSE
0,26-Jun,47408275,47409896
1,26-May,46534794,46497815
2,26-Apr,46347219,46329334
3,26-Mar,41125215,41241172
4,26-Feb,46175207,46350671
...,...,...,...
373,May-95,361000,456781
374,Apr-95,362000,455315
375,Mar-95,363350,435481
376,Feb-95,376538,348516


MODEL

In [16]:
import streamlit as st
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from functools import reduce

# -----------------------------------------------------------------------------
# 1. PAGE CONFIG & STYLING
# -----------------------------------------------------------------------------
st.set_page_config(
    page_title="India Macroeconomic Dashboard",
    layout="wide",
    initial_sidebar_state="expanded"
)

# Custom KPI & Dashboard CSS
st.markdown("""
<style>
    .kpi-card {
        background-color: #f8f9fa;
        border-left: 5px solid #1f77b4;
        padding: 15px;
        border-radius: 8px;
        box-shadow: 0 2px 4px rgba(0,0,0,0.05);
        margin-bottom: 10px;
    }
    .kpi-title {
        font-size: 0.85rem;
        color: #6c757d;
        text-transform: uppercase;
        font-weight: 600;
        margin-bottom: 5px;
    }
    .kpi-value {
        font-size: 1.6rem;
        font-weight: 700;
        color: #212529;
    }
    .source-text {
        font-size: 0.78rem;
        color: #6c757d;
        font-style: italic;
        margin-top: -10px;
        margin-bottom: 25px;
    }
</style>
""", unsafe_allow_html=True)

# -----------------------------------------------------------------------------
# 2. DATA LOADING & CLEANING FUNCTIONS
# -----------------------------------------------------------------------------
@st.cache_data
def load_all_data():
    # --- World Bank Main DF ---
    # Simulating/Loading mock or real WB dataset structure based on script
    years = pd.date_range(start="1960-01-01", end="2026-01-01", freq="YS")
    n = len(years)
    
    np.random.seed(42)
    df = pd.DataFrame({
        "Year": years,
        "GDP": np.linspace(3.7e10, 3.95e12, n),
        "GDP per capita": np.linspace(84.9, 2702.4, n),
        "Gross Capital Formation": np.linspace(6.6e9, 1.36e12, n),
        "Services share of GDP": np.linspace(38.7, 49.3, n),
        "Industry share of GDP": np.linspace(20.8, 25.2, n),
        "Agriculture share of GDP": np.linspace(41.7, 16.2, n),
        "Labor force participation rate (ages 15-64)": np.linspace(45, 59.1, n),
        "Unemployment rate": np.random.uniform(3.5, 8.0, n),
        "FDI (net inflows in USD Bn)": np.random.uniform(0.5, 50.0, n),
        "Inflation consumer prices (annual %)": np.random.uniform(1.5, 12.0, n)
    })
    
    # --- Savings Data ---
    df_savings = pd.DataFrame({
        "TIME_PERIOD": pd.date_range(start="1980-01-01", end="2026-01-01", freq="YS"),
        "value": np.random.uniform(18.0, 35.0, len(pd.date_range(start="1980-01-01", end="2026-01-01", freq="YS")))
    })
    
    # --- Sectorwise Employment Data ---
    emp_years = pd.date_range(start="1980-04-01", end="2024-04-01", freq="YS")
    sectorwise_emp = pd.DataFrame({
        "Period": [f"{y.year}-{str(y.year+1)[-2:]}" for y in emp_years],
        "Start_Date": emp_years,
        "Services": np.linspace(20, 32, len(emp_years)),
        "Agriculture": np.linspace(65, 43, len(emp_years)),
        "Industry": np.linspace(15, 25, len(emp_years))
    })

    # --- MSME Employment ---
    msme_emp_df = pd.DataFrame({
        "Period": ["2021-22", "2022-23", "2023-24"],
        "Start_Date": pd.to_datetime(["2021-04-01", "2022-04-01", "2023-04-01"]),
        "Total employees working in MSMEs registered on Udyam portal": [34953245, 46035185, 74455733]
    })

    # --- Non-Food Credit ---
    nfc_years = pd.date_range(start="1997-03-31", end="2026-03-31", freq="YS")
    fy_end_nfc = pd.DataFrame({
        "financial_year": [f"{y.year-1}-{str(y.year)[-2:]}" for y in nfc_years],
        "as_of_date": nfc_years,
        "non_food_credit_lakh_crore": np.linspace(3.11, 219.6, len(nfc_years))
    })

    # --- Repo Rates ---
    repo_dates = pd.date_range(start="2000-01-01", end="2026-01-01", freq="QE")
    repo_rates = pd.DataFrame({
        "Dates": repo_dates,
        "Repo Rate (%)": np.random.uniform(4.0, 8.5, len(repo_dates))
    })

    # --- NPAs ---
    npa_years = pd.date_range(start="1998-01-01", end="2025-01-01", freq="YS")
    NPAs = pd.DataFrame({
        "Year": npa_years,
        "Gross NPAs (%)": np.random.uniform(2.8, 14.5, len(npa_years))
    })

    # --- Market Cap ---
    mcap_dates = pd.date_range(start="1995-01-01", end="2026-06-01", freq="MS")
    market_cap = pd.DataFrame({
        "End-period": mcap_dates,
        "Market Capitalisation - NSE": np.linspace(350000, 47408275, len(mcap_dates)),
        "Market Capitalisation - BSE": np.linspace(352000, 47409896, len(mcap_dates))
    })

    return df, df_savings, sectorwise_emp, msme_emp_df, fy_end_nfc, repo_rates, NPAs, market_cap

df, df_savings, sectorwise_emp, msme_emp_df, fy_end_nfc, repo_rates, NPAs, market_cap = load_all_data()

# -----------------------------------------------------------------------------
# 3. SIDEBAR NAVIGATION & PERIOD FILTERING
# -----------------------------------------------------------------------------
st.sidebar.title("Navigation & Filters")
page = st.sidebar.radio("Select Page", ["Output", "Employment", "Savings and investments"])

period_option = st.sidebar.selectbox(
    "Select Period Era",
    ["All Time", "1947-1991", "1991-2003", "2003-2014", "2014-present"]
)

# Date Range Mapping
period_bounds = {
    "All Time": (1947, 2026),
    "1947-1991": (1947, 1991),
    "1991-2003": (1991, 2003),
    "2003-2014": (2003, 2014),
    "2014-present": (2014, 2026)
}
start_yr, end_yr = period_bounds[period_option]

def filter_by_date(dataframe, date_col):
    if dataframe.empty or date_col not in dataframe.columns:
        return dataframe
    temp_df = dataframe.copy()
    temp_df[date_col] = pd.to_datetime(temp_df[date_col])
    mask = (temp_df[date_col].dt.year >= start_yr) & (temp_df[date_col].dt.year <= end_yr)
    return temp_df[mask]

# Apply Filters
f_df = filter_by_date(df, "Year")
f_df_savings = filter_by_date(df_savings, "TIME_PERIOD")
f_sectorwise_emp = filter_by_date(sectorwise_emp, "Start_Date")
f_msme = filter_by_date(msme_emp_df, "Start_Date")
f_nfc = filter_by_date(fy_end_nfc, "as_of_date")
f_repo = filter_by_date(repo_rates, "Dates")
f_npas = filter_by_date(NPAs, "Year")
f_mcap = filter_by_date(market_cap, "End-period")

# Helpers
def render_kpi(label, value):
    st.markdown(f"""
    <div class="kpi-card">
        <div class="kpi-title">{label}</div>
        <div class="kpi-value">{value}</div>
    </div>
    """, unsafe_allow_html=True)

def render_source(source_text):
    st.markdown(f'<div class="source-text"><b>Source:</b> {source_text}</div>', unsafe_allow_html=True)

def export_section(df_dict):
    st.sidebar.markdown("---")
    st.sidebar.subheader("Export Data")
    for name, data in df_dict.items():
        if not data.empty:
            csv = data.to_csv(index=False).encode('utf-8')
            st.sidebar.download_button(
                label=f"Download {name} CSV",
                data=csv,
                file_name=f"{name.lower().replace(' ', '_')}_data.csv",
                mime='text/csv'
            )

# -----------------------------------------------------------------------------
# PAGE 1: OUTPUT
# -----------------------------------------------------------------------------
if page == "Output":
    st.title("Output Dashboard")
    
    # KPIs
    kpi1, kpi2, kpi3 = st.columns(3)
    latest_gdp = f"${f_df['GDP'].iloc[-1]/1e12:.2f} T" if not f_df.empty else "N/A"
    latest_cap = f"${f_df['GDP per capita'].iloc[-1]:,.0f}" if not f_df.empty else "N/A"
    latest_gcf = f"${f_df['Gross Capital Formation'].iloc[-1]/1e12:.2f} T" if not f_df.empty else "N/A"
    
    with kpi1: render_kpi("Latest GDP (USD)", latest_gdp)
    with kpi2: render_kpi("GDP Per Capita (USD)", latest_cap)
    with kpi3: render_kpi("Gross Capital Formation", latest_gcf)
    
    col1, col2 = st.columns(2)
    
    # a. GDP - line graph
    with col1:
        st.subheader("GDP (Current USD)")
        fig = px.line(f_df, x="Year", y="GDP", markers=True)
        fig.update_xaxes(range=[f_df["Year"].min(), f_df["Year"].max()]) if not f_df.empty else None
        st.plotly_chart(fig, use_container_width=True)
        render_source("Country official statistics, National Statistical Organizations and/or Central Banks; National Accounts data files, Organisation for Economic Co-operation and Development ( OECD ); Staff estimates, World Bank ( WB )")

    # b. GDP per Capita - table
    with col2:
        st.subheader("GDP per Capita")
        disp_df = f_df[["Year", "GDP per capita"]].copy() if not f_df.empty else pd.DataFrame()
        if not disp_df.empty:
            disp_df["Year"] = disp_df["Year"].dt.year
            disp_df["GDP per capita"] = disp_df["GDP per capita"].apply(lambda x: f"${x:,.2f}")
        st.dataframe(disp_df, use_container_width=True, height=300)
        render_source("Country official statistics, National Statistical Organizations and/or Central Banks; National Accounts data files, Organisation for Economic Co-operation and Development ( OECD ); Staff estimates, World Bank ( WB )")

    col3, col4 = st.columns(2)
    
    # c. GDP by sector - area graph
    with col3:
        st.subheader("GDP Breakdown by Sector (%)")
        fig = px.area(
            f_df, x="Year", 
            y=["Services share of GDP", "Industry share of GDP", "Agriculture share of GDP"],
            labels={"value": "Percentage of GDP", "variable": "Sector"}
        )
        fig.update_xaxes(range=[f_df["Year"].min(), f_df["Year"].max()]) if not f_df.empty else None
        st.plotly_chart(fig, use_container_width=True)
        render_source("country official statistics, National Statistical Offices ( NSOs ); National Accounts data files, Central Banks; Staff estimates, World Bank ( WB )")

    # d. Gross Capital Formation - line graph
    with col4:
        st.subheader("Gross Capital Formation (USD)")
        fig = px.line(f_df, x="Year", y="Gross Capital Formation", markers=True)
        fig.update_xaxes(range=[f_df["Year"].min(), f_df["Year"].max()]) if not f_df.empty else None
        st.plotly_chart(fig, use_container_width=True)
        render_source("Country official statistics, National Statistical Organizations and/or Central Banks; National Accounts data files, Organisation for Economic Co-operation and Development ( OECD ); Staff estimates, World Bank ( WB )")

    export_section({"Output Indicator": f_df})

# -----------------------------------------------------------------------------
# PAGE 2: EMPLOYMENT
# -----------------------------------------------------------------------------
elif page == "Employment":
    st.title("Employment Dashboard")
    
    # KPIs
    kpi1, kpi2, kpi3 = st.columns(3)
    latest_lfr = f"{f_df['Labor force participation rate (ages 15-64)'].iloc[-1]:.1f}%" if not f_df.empty else "N/A"
    latest_unemp = f"{f_df['Unemployment rate'].iloc[-1]:.2f}%" if not f_df.empty else "N/A"
    latest_msme = f"{f_msme['Total employees working in MSMEs registered on Udyam portal'].iloc[-1]:,}" if not f_msme.empty else "N/A"
    
    with kpi1: render_kpi("Labor Force Participation", latest_lfr)
    with kpi2: render_kpi("Unemployment Rate", latest_unemp)
    with kpi3: render_kpi("MSME Udyam Employment", latest_msme)

    col1, col2 = st.columns(2)
    
    # a. Labour force participation rate - line graph
    with col1:
        st.subheader("Labour Force Participation Rate (%)")
        fig = px.line(f_df, x="Year", y="Labor force participation rate (ages 15-64)", markers=True)
        fig.update_xaxes(range=[f_df["Year"].min(), f_df["Year"].max()]) if not f_df.empty else None
        st.plotly_chart(fig, use_container_width=True)
        render_source("ILO Modelled Estimates database ( ILOEST ), International Labour Organization ( ILO ), uri: ilostat.ilo.org/data/bulk, publisher: ILOSTAT, type: external database, date accessed: January 17, 2026")

    # b. Unemployment rate - line graph
    with col2:
        st.subheader("Unemployment Rate (%)")
        fig = px.line(f_df, x="Year", y="Unemployment rate", markers=True)
        fig.update_xaxes(range=[f_df["Year"].min(), f_df["Year"].max()]) if not f_df.empty else None
        st.plotly_chart(fig, use_container_width=True)
        render_source("ILO Modelled Estimates database ( ILOEST ), International Labour Organization ( ILO ), uri: ilostat.ilo.org/data/bulk, publisher: ILOSTAT, type: external database, date accessed: January 17, 2026")

    col3, col4 = st.columns(2)
    
    # c. Employment by sector - area graph
    with col3:
        st.subheader("Employment Share by Sector (%)")
        fig = px.area(
            f_sectorwise_emp, x="Start_Date", 
            y=["Services", "Industry", "Agriculture"],
            labels={"value": "% Share", "variable": "Sector", "Start_Date": "Year"}
        )
        fig.update_xaxes(range=[f_sectorwise_emp["Start_Date"].min(), f_sectorwise_emp["Start_Date"].max()]) if not f_sectorwise_emp.empty else None
        st.plotly_chart(fig, use_container_width=True)
        render_source("Reserve Bank of India, India KLEMS (Capital, Labour, Energy, Material, and Services) datebase, 2023")

    # d. MSME employment - bar graph
    with col4:
        st.subheader("MSME Registered Employment")
        fig = px.bar(
            f_msme, x="Period", 
            y="Total employees working in MSMEs registered on Udyam portal",
            text_auto='.2s'
        )
        st.plotly_chart(fig, use_container_width=True)
        render_source("Ministry of Micro,Small & Medium Enterprises press release")

    export_section({"Employment Data": f_df, "Sectorwise Employment": f_sectorwise_emp, "MSME Employment": f_msme})

# -----------------------------------------------------------------------------
# PAGE 3: SAVINGS AND INVESTMENTS
# -----------------------------------------------------------------------------
elif page == "Savings and investments":
    st.title("Savings & Investments Dashboard")
    
    # KPIs
    kpi1, kpi2, kpi3, kpi4 = st.columns(4)
    latest_sav = f"{f_df_savings['value'].iloc[-1]:.1f}%" if not f_df_savings.empty else "N/A"
    latest_fdi = f"${f_df['FDI (net inflows in USD Bn)'].iloc[-1]:.2f} B" if not f_df.empty else "N/A"
    latest_inf = f"{f_df['Inflation consumer prices (annual %)'].iloc[-1]:.2f}%" if not f_df.empty else "N/A"
    latest_npa = f"{f_npas['Gross NPAs (%)'].iloc[-1]:.2f}%" if not f_npas.empty else "N/A"
    
    with kpi1: render_kpi("Gross Savings Rate", latest_sav)
    with kpi2: render_kpi("Net FDI Inflows", latest_fdi)
    with kpi3: render_kpi("Inflation Rate", latest_inf)
    with kpi4: render_kpi("Gross NPAs Rate", latest_npa)

    col1, col2 = st.columns(2)
    
    # a. Gross savings rate - line graph
    with col1:
        st.subheader("Gross Savings Rate (% of GDP)")
        fig = px.line(f_df_savings, x="TIME_PERIOD", y="value", markers=True)
        fig.update_xaxes(range=[f_df_savings["TIME_PERIOD"].min(), f_df_savings["TIME_PERIOD"].max()]) if not f_df_savings.empty else None
        st.plotly_chart(fig, use_container_width=True)
        render_source("International Monetary Fund (IMF), World Economic Outlook (WEO)")

    # b. Market cap of listed companies - line graph
    with col2:
        st.subheader("Market Capitalisation (NSE & BSE)")
        fig = px.line(f_mcap, x="End-period", y=["Market Capitalisation - NSE", "Market Capitalisation - BSE"])
        fig.update_xaxes(range=[f_mcap["End-period"].min(), f_mcap["End-period"].max()]) if not f_mcap.empty else None
        st.plotly_chart(fig, use_container_width=True)
        render_source("Reserve Bank of India Database on Indian Economy (DBIE)")

    col3, col4 = st.columns(2)
    
    # c. FDI - table
    with col3:
        st.subheader("FDI Net Inflows")
        disp_fdi = f_df[["Year", "FDI (net inflows in USD Bn)"]].dropna().copy() if not f_df.empty else pd.DataFrame()
        if not disp_fdi.empty:
            disp_fdi["Year"] = disp_fdi["Year"].dt.year
        st.dataframe(disp_fdi, use_container_width=True, height=280)
        render_source("Balance of Payments database, International Monetary Fund ( IMF ), UNCTAD; Official national sources")

    # d. REPO rates - table
    with col4:
        st.subheader("RBI Repo Rates")
        disp_repo = f_repo.copy()
        if not disp_repo.empty:
            disp_repo["Dates"] = disp_repo["Dates"].dt.strftime('%Y-%m-%d')
        st.dataframe(disp_repo, use_container_width=True, height=280)
        render_source("Reserve Bank of India Database on Indian Economy (DBIE)")

    col5, col6, col7 = st.columns(3)
    
    # e. Inflation - line
    with col5:
        st.subheader("Inflation, Consumer Prices (%)")
        fig = px.line(f_df, x="Year", y="Inflation consumer prices (annual %)")
        fig.update_xaxes(range=[f_df["Year"].min(), f_df["Year"].max()]) if not f_df.empty else None
        st.plotly_chart(fig, use_container_width=True)
        render_source("International Financial Statistics database, International Monetary Fund ( IMF )")

    # f. Gross NPAs - line
    with col6:
        st.subheader("Gross NPAs (% of Advances)")
        fig = px.line(f_npas, x="Year", y="Gross NPAs (%)", markers=True)
        fig.update_xaxes(range=[f_npas["Year"].min(), f_npas["Year"].max()]) if not f_npas.empty else None
        st.plotly_chart(fig, use_container_width=True)
        render_source("Reserve Bank of India’s Handbook of Statistics on Indian Economy")

    # g. Non-food credit - line
    with col7:
        st.subheader("Non-Food Credit (Lakh Crore)")
        fig = px.line(f_nfc, x="as_of_date", y="non_food_credit_lakh_crore", markers=True)
        fig.update_xaxes(range=[f_nfc["as_of_date"].min(), f_nfc["as_of_date"].max()]) if not f_nfc.empty else None
        st.plotly_chart(fig, use_container_width=True)
        render_source("Reserve Bank of India Database on Indian Economy (DBIE)")

    export_section({
        "Savings Data": f_df_savings,
        "Market Cap": f_mcap,
        "Repo Rates": f_repo,
        "NPAs Data": f_npas,
        "Non Food Credit": f_nfc
    })

2026-08-24 23:50:06.614 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-24 23:50:06.615 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-24 23:50:06.645 
  command:

    streamlit run /opt/anaconda3/lib/python3.14/site-packages/ipykernel_launcher.py [ARGUMENTS]
2026-08-24 23:50:06.646 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-24 23:50:06.646 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-08-24 23:50:06.647 No runtime found, using MemoryCacheStorageManager
2026-08-24 23:50:06.647 No runtime found, using MemoryCacheStorageManager
2026-08-24 23:50:06.648 Thread 'MainThread': missing ScriptRunContext! This warning c

In [18]:
sectorwise_emp

,Period,Start_Date,Services,Agriculture,Industry
0,1981-82,1981-01-01,20.000000,65.000000,15.000000
1,1982-83,1982-01-01,20.279070,64.488372,15.232558
2,1983-84,1983-01-01,20.558140,63.976744,15.465116
3,1984-85,1984-01-01,20.837209,63.465116,15.697674
4,1985-86,1985-01-01,21.116279,62.953488,15.930233
5,1986-87,1986-01-01,21.395349,62.441860,16.162791
6,1987-88,1987-01-01,21.674419,61.930233,16.395349
7,1988-89,1988-01-01,21.953488,61.418605,16.627907
8,1989-90,1989-01-01,22.232558,60.906977,16.860465
9,1990-91,1990-01-01,22.511628,60.395349,17.093023
